# Import libraries

In [27]:
import os
import re
from dotenv import load_dotenv
from itertools import groupby
from functools import partial
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_classic.indexes import SQLRecordManager, index
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings

load_dotenv()

True

# Get original files and split into small chunks

**Important:**

- The files format must be .pdf.
- The name of the files must be: 'book_chapter_(chapter).pdf

In [28]:
# Load all PDFs from data folder
all_docs = []
data_dir = "data"
files = sorted([f for f in os.listdir(data_dir) if f.endswith(".pdf")])

for file in files:
    # Extract chapter number from filename (e.g., book-chapter-00.pdf -> 00)
    match = re.search(r"book_chapter_(\d+)\.pdf", file)
    if match:
        chapter_num = match.group(1)
        loader = PyPDFLoader(os.path.join(data_dir, file))
        docs = loader.load()
        
        # Add metadata to each document (page)
        for doc in docs:
            doc.metadata.update({
                "volume": 1,
                "chapter": int(chapter_num)
            })
        all_docs.extend(docs)

# Split into smaller chunks
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,     # Average characters per chunk
    chunk_overlap=75,   # Overlap for context across chunks
    add_start_index=True # Add chunk index in metadata for traceability
)
chunks = child_splitter.split_documents(all_docs)

In [29]:
print(f"Split into {len(chunks)} chunks")
print("keys:")
# print(vars(chunks[0]).keys())  # First chunk
print("metadata sample (first chunk):") 
print(chunks[0].metadata)
print("metadata sample (last chunk):")
print(chunks[-1].metadata)
print("page_content sample:")
print(chunks[0].page_content[:200])

Split into 461 chunks
keys:
metadata sample (first chunk):
{'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20260329105746', 'source': 'data\\book_chapter_00.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1', 'volume': 1, 'chapter': 0, 'start_index': 0}
metadata sample (last chunk):
{'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20260329110040', 'source': 'data\\book_chapter_03.pdf', 'total_pages': 24, 'page': 23, 'page_label': '24', 'volume': 1, 'chapter': 3, 'start_index': 433}
page_content sample:
Speech and Language Processing
An Introduction to Natural Language Processing,
Computational Linguistics, and Speech Recognition
with Language Models
Third Edition draft
Daniel Jurafsky
Stanford Unive


## Keep only useful metadata 

In [30]:
keys_to_keep =["creationdate", "source", "total_pages", "page", "volume", "chapter", "start_index"]

for chunk in chunks:
    # Filter the dictionary
    chunk.metadata = {k: v for k, v in chunk.metadata.items() if k in keys_to_keep}

## Analyze the chunks

Group chunks by chapter

In [31]:
def group_docs_by_metadata(documents, key="source"):
    # Sort is required for groupby to work correctly
    sorted_docs = sorted(documents, key=lambda x: x.metadata.get(key, "unknown"))
    
    grouped = {
        k: list(g) 
        for k, g in groupby(sorted_docs, key=lambda x: x.metadata.get(key, "unknown"))
    }
    return grouped

In [32]:
chunks_by_chapter = group_docs_by_metadata(chunks, "chapter")
for chapter in chunks_by_chapter:
    print(f"Chunks of the chapter {chapter}: {len(chunks_by_chapter[chapter])}")

Chunks of the chapter 0: 52
Chunks of the chapter 1: 3
Chunks of the chapter 2: 244
Chunks of the chapter 3: 162


## Index

In [33]:
api_key = os.getenv("API_KEY")
embedding = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001", api_key=api_key)

In [34]:
get_vectorstore = partial(Chroma, embedding_function=embedding)

In [35]:
collection_name = "langchain_docs_index"

In [36]:
namespace = f"chroma/{collection_name}"
record_manager = SQLRecordManager(
    namespace=namespace,
    db_url="sqlite:///:memory:",
)
record_manager.create_schema()

In [37]:
vector_store = get_vectorstore(
    collection_name=collection_name,
)

Test the indexing with only 3 chunks

In [38]:
index(
    docs_source=chunks[0:3],
    record_manager=record_manager,
    vector_store=vector_store,
    source_id_key="source",
    cleanup="incremental",
)

{'num_added': 3, 'num_updated': 0, 'num_skipped': 0, 'num_deleted': 0}

In [39]:
# Testing incremental cleanup, trying to add the same chunks
index(
    docs_source=chunks[0:3],
    record_manager=record_manager,
    vector_store=vector_store,
    source_id_key="source",
    cleanup="incremental",
)

{'num_added': 0, 'num_updated': 0, 'num_skipped': 3, 'num_deleted': 0}